# Synthetic Data Generator
### Step 11 : Build Synthetic Final Aligned Output Stage

This notebook builds the final Kafka/Postgres synthetic output so it matches the Kaggle-style dataset shape:

- `dataset_id`
- `run_id`
- `asset_id`
- `timestamp`
- `sensor_00` ... `sensor_51`
- `machine_status`

It uses the rebuilt stage as the source table and does **not** read premelt for this final output step.


In [1]:
from utils.postgres_util import (
    get_engine_from_env, 
    read_sql_dataframe,
)

from utils.synthetic.generator.synthetic_final_aligned_output_writer import build_synthetic_final_aligned_output_stage

In [2]:
SCHEMA = str("capstone")

DATASET_ID = str("pump_synthetic_v1")
RUN_ID = str("premelt_run_001")

NUMBER_OF_SENSORS = int(52)

IF_EXISTS_FLAG = str("replace")
COMPLETE_ONLY_FLAG = True
OBSERVATION_WINDOW_SIZE = int(2500)

REBUILT_SOURCE_TABLE_NAME = str("synthetic_sensor_observations_rebuilt_stage")
TARGET_TABLE_NAME = str("synthetic_sensor_observations_final_output")

TIMESTAMP_SOURCE_PRIORITY = (
    "observation_timestamp",
    "timestamp",
    "created_at",
)

STATUS_SOURCE_PRIORITY = (
    "machine_status",
    "stream_state",
    "phase",
)

STATUS_MAPPING = {
    "normal": "NORMAL",
    "broken": "BROKEN",
    "abnormal": "BROKEN",
    "failure": "BROKEN",
    "failed": "BROKEN",
    "fault": "BROKEN",
    "recovering": "RECOVERING",
    "recovery": "RECOVERING",
}


---

In [3]:
engine = get_engine_from_env()

---

In [4]:
final_output_result = build_synthetic_final_aligned_output_stage(
    engine=engine,
    schema=SCHEMA,
    rebuilt_table=REBUILT_SOURCE_TABLE_NAME,
    target_table=TARGET_TABLE_NAME,
    dataset_id=DATASET_ID,
    run_id=RUN_ID,
    n_sensors=NUMBER_OF_SENSORS,
    complete_only=COMPLETE_ONLY_FLAG,
    if_exists=IF_EXISTS_FLAG,
    observation_window_size=OBSERVATION_WINDOW_SIZE,
    timestamp_source_priority=TIMESTAMP_SOURCE_PRIORITY,
    status_source_priority=STATUS_SOURCE_PRIORITY,
    status_mapping=STATUS_MAPPING,
    timestamp_output_column="timestamp",
    machine_status_output_column="machine_status",
)

display(final_output_result)

{'status': 'built',
 'dataset_id': 'pump_synthetic_v1',
 'run_id': 'premelt_run_001',
 'rebuilt_rows_available': 72000,
 'rebuilt_rows_read': 72000,
 'final_rows_written': 72000,
 'windows_processed': 29,
 'target_table': 'synthetic_sensor_observations_final_output',
 'timestamp_output_column': 'timestamp',
 'machine_status_output_column': 'machine_status'}

----

In [5]:
validation_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        COUNT(*) AS row_count,
        COUNT(DISTINCT dataset_id) AS dataset_id_count,
        COUNT(DISTINCT run_id) AS run_id_count,
        COUNT(DISTINCT asset_id) AS asset_id_count,
        MIN(timestamp) AS min_timestamp,
        MAX(timestamp) AS max_timestamp,
        COUNT(*) FILTER (WHERE machine_status = 'NORMAL') AS normal_rows,
        COUNT(*) FILTER (WHERE machine_status = 'BROKEN') AS broken_rows,
        COUNT(*) FILTER (WHERE machine_status = 'RECOVERING') AS recovering_rows
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    """
)

display(validation_dataframe)


,row_count,dataset_id_count,run_id_count,asset_id_count,min_timestamp,max_timestamp,normal_rows,broken_rows,recovering_rows
0,72000,1,1,1,2026-03-19 08:00:00+00:00,2026-05-08 07:59:00+00:00,68796,4,3200


---


### Sample Output

This shows the final Kaggle-shaped output table layout.


In [12]:
sample_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        dataset_id,
        run_id,
        asset_id,
        timestamp,
        sensor_00,
        sensor_01,
        sensor_02,
        sensor_03,
        sensor_04,
        machine_status
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    ORDER BY timestamp, dataset_id, run_id, asset_id
    LIMIT 10
    """
)

display(sample_dataframe)


,dataset_id,run_id,asset_id,timestamp,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,machine_status
0,pump_synthetic_v1,premelt_run_001,pump_asset_001,2026-03-19 08:00:00+00:00,2.339929,48.423782,52.319363,45.037003,609.487671,NORMAL
1,pump_synthetic_v1,premelt_run_001,pump_asset_001,2026-03-19 08:00:01+00:00,2.366877,48.430668,51.977501,44.732082,615.093567,NORMAL
2,pump_synthetic_v1,premelt_run_001,pump_asset_001,2026-03-19 08:00:02+00:00,2.418399,48.400005,52.017746,44.741520,625.548401,NORMAL
3,pump_synthetic_v1,premelt_run_001,pump_asset_001,2026-03-19 08:00:03+00:00,2.419097,48.411598,52.030724,44.729973,625.465088,NORMAL
4,pump_synthetic_v1,premelt_run_001,pump_asset_001,2026-03-19 08:00:04+00:00,2.415245,48.390038,51.796700,44.507263,625.718994,NORMAL
5,pump_synthetic_v1,premelt_run_001,pump_asset_001,2026-03-19 08:00:05+00:00,2.407531,48.400581,51.719734,44.428181,624.135315,NORMAL
6,pump_synthetic_v1,premelt_run_001,pump_asset_001,2026-03-19 08:00:06+00:00,2.398601,48.378510,51.699894,44.434540,621.922119,NORMAL
7,pump_synthetic_v1,premelt_run_001,pump_asset_001,2026-03-19 08:00:07+00:00,2.398482,48.363518,51.926392,44.639126,620.877197,NORMAL
8,pump_synthetic_v1,premelt_run_001,pump_asset_001,2026-03-19 08:00:08+00:00,2.420591,48.354713,51.802505,44.497536,625.323914,NORMAL
9,pump_synthetic_v1,premelt_run_001,pump_asset_001,2026-03-19 08:00:09+00:00,2.420582,48.394501,52.303089,44.911942,625.518494,NORMAL


---

In [13]:
status_distribution_dataframe = read_sql_dataframe(
    engine,
    f"""
    SELECT
        machine_status,
        COUNT(*) AS row_count
    FROM {SCHEMA}.{TARGET_TABLE_NAME}
    GROUP BY machine_status
    ORDER BY machine_status
    """
)

display(status_distribution_dataframe)


,machine_status,row_count
0,BROKEN,72
1,BUILDUP,24939
2,NORMAL,667389
3,RECOVERING,57600


---

In [ ]:
# Dispose Engine
engine.dispose()